# Notebook 3 — Huấn luyện & So sánh: YOLOv8 vs Faster R-CNN

**Bài tập lớn số 2 · CO5085 · HCMUT 2025-2026**

## Mục tiêu
1. Hiểu kiến trúc One-Stage (YOLO) vs Two-Stage (Faster R-CNN)
2. Fine-tune cả hai mô hình trên Pascal VOC 2012
3. So sánh quá trình training và kết quả

## Cài đặt flag
Đặt `TRAIN_MODE = True` để chạy training thực sự.
Đặt `TRAIN_MODE = False` để load kết quả đã có và chạy nhanh.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

TRAIN_MODE = False  # Đổi thành True để chạy training thực sự

import torch
from src.data import get_frcnn_loaders, get_device
from src.models import get_faster_rcnn, get_yolov8, get_model_info
from src.train import fit_frcnn, train_yolov8, load_frcnn_checkpoint
from src.utils import print_detection_results_table, plot_loss_curves, load_metrics_json

device = get_device()
print("Device:", device)

## 1. Kiến trúc: One-Stage vs Two-Stage

### YOLOv8 (One-Stage)
```
Input image
    ↓
CSPDarknet backbone (feature extraction)
    ↓
FPN neck (multi-scale features)
    ↓
Detection head (đồng thời: classification + localization)
    ↓
Output: [N, 4+num_classes] per anchor-free grid cell
```
**Ưu điểm:** Nhanh (single forward pass)
**Nhược điểm:** Đôi khi kém chính xác hơn với small objects

### Faster R-CNN (Two-Stage)
```
Input image
    ↓
ResNet-50 backbone + FPN (feature extraction)
    ↓
RPN (Region Proposal Network) → ~300 proposal boxes
    ↓
ROI Align (crop features cho từng proposal)
    ↓
Box Head (classification + box regression)
    ↓
Output: detected objects với class và bbox
```
**Ưu điểm:** Chính xác hơn, đặc biệt với overlapping objects
**Nhược điểm:** Chậm hơn (two forward passes)

In [ ]:
# Khởi tạo và so sánh mô hình
print("=== Model Info ===")
try:
    yolo = get_yolov8('n', pretrained=True)
    yolo_info = get_model_info(yolo, 'YOLOv8n')
    print(f"YOLOv8n:         {yolo_info['total_params']} params")
except Exception as e:
    print(f"YOLOv8n: không load được ({e})")

frcnn = get_faster_rcnn(num_classes=21, pretrained_backbone=True)
frcnn_info = get_model_info(frcnn, 'Faster R-CNN ResNet-50 FPN')
print(f"Faster R-CNN:    {frcnn_info['total_params']} params")

## 2. Faster R-CNN — Loss Structure

In [ ]:
# Demo: Faster R-CNN trả về dict losses (không phải logits)
from src.data import VOCDetectionDataset, get_val_transforms, collate_fn
from torch.utils.data import DataLoader

ds = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                          transforms=get_val_transforms())
loader = DataLoader(ds, batch_size=2, collate_fn=collate_fn)

frcnn.train()
frcnn.to(device)
images, targets = next(iter(loader))
images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

with torch.no_grad():
    loss_dict = frcnn(images, targets)

print("Loss components từ Faster R-CNN:")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")
print(f"  TOTAL: {sum(v.item() for v in loss_dict.values()):.4f}")

## 3. Training

In [ ]:
if TRAIN_MODE:
    # Fine-tune Faster R-CNN
    train_loader, val_loader = get_frcnn_loaders('../data/voc', batch_size=4, num_workers=2)
    model = get_faster_rcnn(num_classes=21)
    config = {'epochs': 10, 'lr': 0.005, 'device': device,
               'save_path': '../results/checkpoints/frcnn_voc.pth'}
    history = fit_frcnn(model, train_loader, val_loader, config)
    plot_loss_curves(history, 'Faster R-CNN Training Loss',
                     save_path='../results/plots/frcnn_loss.png')
else:
    print("[TRAIN_MODE=False] Bỏ qua training. Load kết quả đã có...")
    try:
        frcnn_results = load_metrics_json('../results/metrics/frcnn_results.json')
        if 'history' in frcnn_results:
            plot_loss_curves(frcnn_results['history'], 'Faster R-CNN Training Loss (loaded)')
    except FileNotFoundError:
        print("  Chưa có kết quả. Chạy: python scripts/run_finetune_frcnn.py")

In [ ]:
if TRAIN_MODE:
    # Fine-tune YOLOv8
    from src.data import prepare_yolo_dataset
    yaml_path = prepare_yolo_dataset('../data/voc', '../data/voc_yolo')
    best_pt = train_yolov8(yaml_path, model_size='n', epochs=2,
                            project='../results', name='yolov8n_voc')
    print(f"YOLOv8 best checkpoint: {best_pt}")
else:
    print("[TRAIN_MODE=False] Bỏ qua YOLOv8 training.")

## 4. Kết quả nhanh

In [ ]:
# Load và hiển thị kết quả từ file JSON
try:
    yolo_r = load_metrics_json('../results/metrics/yolo_results.json')
    frcnn_r = load_metrics_json('../results/metrics/frcnn_results.json')
    results = {
        yolo_r.get('model', 'YOLOv8n'): yolo_r,
        frcnn_r.get('model', 'Faster R-CNN'): frcnn_r,
    }
    print_detection_results_table(results)
except FileNotFoundError:
    print("Chưa có kết quả. Cần chạy training trước.")
    print("  python scripts/run_finetune_yolo.py")
    print("  python scripts/run_finetune_frcnn.py")